In [1]:
import pandas as pd
import numpy as np

### Reading in master hospital table:

In [2]:
hospitals_master = pd.read_csv('../data/hospitals_master.csv')

In [3]:
hospitals_master.shape

(851, 18)

In [4]:
hospitals_master.columns

Index(['CCN', 'Facility Name', 'State', 'Zip', 'County', 'Emergency Services',
       'Hospital Ownership', 'Closed', 'Closure Date', 'Full Address',
       'Prior Name', 'RUCA', 'Tract_Code', 'CBSA_Code', 'CBSA_Title',
       'Metro_Status', 'County_FIPS', 'State_FIPS'],
      dtype='object')

In [5]:
hospitals_master['Closed'].value_counts()

Closed
0    813
1     20
2     18
Name: count, dtype: int64

I'll need to filter out any hospitals that closed before 2010, since the financial data I'll be using only goes back to 2010. This will affect 5 hospitals (13% of the closed or converted hospitals in the dataset).

In [6]:
hospitals_master['Closure Date'] = pd.to_datetime(hospitals_master['Closure Date'])

In [7]:
hospitals_master[~hospitals_master['Closure Date'].isna()]['Closure Date'].sort_values()

367   2005-03-01
511   2005-10-01
191   2005-12-01
558   2007-11-01
582   2009-11-01
249   2011-03-01
580   2012-03-01
212   2014-10-01
539   2015-08-01
707   2016-01-01
485   2020-01-01
657   2020-08-01
532   2020-10-01
677   2020-10-01
179   2021-06-01
464   2021-07-01
253   2022-01-01
2     2022-02-01
92    2022-09-01
32    2022-12-01
783   2023-03-01
705   2023-06-01
416   2023-08-01
339   2023-10-01
312   2023-10-01
681   2023-10-01
706   2023-10-01
355   2024-03-01
36    2024-04-01
182   2024-05-01
465   2024-09-01
794   2025-01-01
528   2025-05-01
269   2025-09-01
161   2025-10-01
100   2026-05-01
Name: Closure Date, dtype: datetime64[ns]

In [8]:
hospitals_master = hospitals_master[~(hospitals_master['Closure Date'] < '2010-01-01')]

In [9]:
hospitals_master.isna().sum()

CCN                     0
Facility Name           0
State                   0
Zip                     0
County                  0
Emergency Services      0
Hospital Ownership      0
Closed                  0
Closure Date          815
Full Address            0
Prior Name            844
RUCA                    0
Tract_Code              0
CBSA_Code             196
CBSA_Title            196
Metro_Status            0
County_FIPS             0
State_FIPS              0
dtype: int64

In [10]:
hospitals_master.head()

,CCN,Facility Name,State,Zip,County,Emergency Services,Hospital Ownership,Closed,Closure Date,Full Address,Prior Name,RUCA,Tract_Code,CBSA_Code,CBSA_Title,Metro_Status,County_FIPS,State_FIPS
0,190044,ACADIA GENERAL HOSPITAL,LA,70526,ACADIA,Yes,Voluntary non-profit - Private,0,NaT,"1305 CROWLEY RAYNE HIGHWAY, CROWLEY, LA, 70526",NaN,4.0,960801.0,29180.0,"Lafayette, LA",Metropolitan Statistical Area,22001,22.0
1,390163,ACMH HOSPITAL,PA,16201,ARMSTRONG,Yes,Voluntary non-profit - Private,0,NaT,"1 NOLTE DRIVE, KITTANNING, PA, 16201",NaN,4.0,950500.0,38300.0,"Pittsburgh, PA",Metropolitan Statistical Area,42005,42.0
2,320070,ACOMA-CANONCITO-LAGUNA SERVICE UNIT,NM,87034,CIBOLA,Yes,"Governmental, Federal",1,2022-02-01,"80B VETERANS BLVD, ACOMA, NM, 87034",NaN,2.0,941500.0,NaN,NaN,Neither,35006,35.0
3,360159,ADENA REGIONAL MEDICAL CENTER,OH,45601,ROSS,Yes,Voluntary non-profit - Private,0,NaT,"272 HOSPITAL ROAD, CHILLICOTHE, OH, 45601",NaN,4.0,956300.0,17060.0,"Chillicothe, OH",Micropolitan Statistical Area,39141,39.0
4,330079,ADIRONDACK MEDICAL CENTER - SARANAC LAKE,NY,12983,FRANKLIN,Yes,Voluntary non-profit - Other,0,NaT,"2233 STATE ROUTE 86, SARANAC LAKE, NY, 12983",NaN,7.0,950900.0,NaN,NaN,Neither,36033,36.0


### Gathering hospital financial data:

Hospital financial data aggregation was performed in the "Financial Report Data Gathering Notebook" notebook with the help of code drawn from the HCRIS-databuilder GitHub project (https://github.com/Rush-Quality-Analytics/HCRIS-databuilder/tree/master).

In [11]:
financials = pd.read_csv('../data/HCRIS_databuilder/filtered_datasets/HCRIS_filtered.csv').drop(columns=['Report_Period_Begin_Yr','Urban (1) and Rural (2) Indicator','State']).rename(columns={'Beginning FFY':'Year',': General Ownership Type':'General Ownership Type'})

                                                                                                         

In [12]:
financials.columns

Index(['Hospital', 'Hospital type', 'Control type', 'Medicaid charges',
       'HAC reduction adjustment amount',
       'STATEMENT OF REVENUES AND EXPENSES: Net Income (G3_C1_29)',
       'ADJUSTED SALARIES, Subtotal Salaries',
       'BALANCE SHEET: Prepaid expenses (G_C1THRU4_8)',
       'BED DAYS: Total Hospital', 'Number of Interns & Residents',
       'BALANCE SHEET: Cash on Hand and in Banks (G_C1THRU4_1)',
       'Cost To Charge Ratio', 'Total discharges',
       'REIMBURSEMENT SETTLEMENT: Payment to cost ratio',
       'Cost of Uncompensated Care',
       'BALANCE SHEET: Accounts Receivable (G_C1THRU4_4)', 'Total Salaries',
       'BALANCE SHEET: Total Assets (G_C1THRU4_36)',
       'BALANCE SHEET: Total Current Assets (G_C1THRU4_11)',
       'REIMBURSEMENT SETTLEMENT: Interim payments', 'Total Bad Debt expense',
       'TRIAL BALANCE OF EXPENSE ACCOUNTS: Interest Expense (A_C2_113)',
       'NUMBER OF BEDS: Adults & Pediatrics',
       'BALANCE SHEET: Temporary Investments (G

In [13]:
# Separating Hospital column into Facility Name and CCN, then dropping Hospital column
financials[['Facility Name', 'CCN']] = financials['Hospital'].str.extract(r'(.*)\s+\((\d+)\)')
financials = financials.drop(columns='Hospital')

In [14]:
# Creating a report period column to use later
financials['Fiscal Year End Date'] = pd.to_datetime(financials['Fiscal Year End Date'])
financials['Fiscal Year Begin Date'] = pd.to_datetime(financials['Fiscal Year Begin Date'])
financials['Report Period (Months)'] = (financials['Fiscal Year End Date'].dt.to_period('M') - financials['Fiscal Year Begin Date'].dt.to_period('M')).apply(lambda x: x.n)


In [15]:
financials.head(2)

,Hospital type,Control type,Medicaid charges,HAC reduction adjustment amount,STATEMENT OF REVENUES AND EXPENSES: Net Income (G3_C1_29),"ADJUSTED SALARIES, Subtotal Salaries",BALANCE SHEET: Prepaid expenses (G_C1THRU4_8),BED DAYS: Total Hospital,Number of Interns & Residents,BALANCE SHEET: Cash on Hand and in Banks (G_C1THRU4_1),...,Financial Indicators: SOLVENCY Debt-to-Equity Ratio,Financial Indicators: SOLVENCY Debt Ratio,Financial Indicators: SOLVENCY Equity Ratio,Financial Indicators: SOLVENCY Interest Coverage Ratio,Financial Indicators: SOLVENCY total assets less total liabilities,Financial Indicators: EFFICIENCY asset turnover ratio,Financial Indicators: EFFICIENCY Accounts Receivable Turnover Ratio,Facility Name,CCN,Report Period (Months)
0,General Short Term (Acute Care Hospitals),Voluntary Nonprofit-Other,2937126.0,NaN,-1896638.0,4854804.0,NaN,17080.0,167.81,629080.0,...,1.121873,0.528718,0.471282,NaN,5374708.0,0.795823,1.255900,ACADIA GENERAL HOSPITAL,190044,3
1,General Short Term (Acute Care Hospitals),Voluntary Nonprofit-Other,10731317.0,NaN,-4070558.0,15097743.0,NaN,51100.0,354.90,464305.0,...,2.131645,0.680679,0.319321,NaN,3154921.0,3.627023,5.701668,ACADIA GENERAL HOSPITAL,190044,11


Ensuring we have one row per hospital per year:

In [16]:
financials[financials[['Facility Name','Year']].duplicated(keep=False)][['Facility Name','Year','Fiscal Year Begin Date','Fiscal Year End Date','Total Bad Debt expense','Total cost of charity care']]


,Facility Name,Year,Fiscal Year Begin Date,Fiscal Year End Date,Total Bad Debt expense,Total cost of charity care
235,AMERICAN HEALTHCARE SYSTEMS,2022,2021-10-01,2021-12-31,3738257.0,1654222.0
236,AMERICAN HEALTHCARE SYSTEMS,2022,2022-01-01,2022-12-31,27475472.0,7585520.0
427,ATHENS LIMESTONE,2016,2016-01-01,2016-06-30,11291863.0,159029.0
428,ATHENS LIMESTONE,2016,2016-07-01,2017-06-30,20210963.0,524217.0
487,ATRIUM HEALTH NAVICENT BALDWIN,2021,2021-01-01,2021-07-02,4801970.0,1578971.0
...,...,...,...,...,...,...
12031,WRMC HOSPITAL OPERATING CORPORATION,2018,2018-07-01,2019-06-30,7664715.0,2274174.0
12075,YAMPA VALLEY MEDICAL CENTER,2018,2017-10-01,2018-06-30,2191748.0,651822.0
12076,YAMPA VALLEY MEDICAL CENTER,2018,2018-07-01,2019-06-30,54973.0,1128837.0
12082,YAMPA VALLEY MEDICAL CENTER,2024,2023-10-01,2024-06-30,NaN,NaN


We see that some rows have duplicated hospital-years. This is either because "(1) each cost report covers part of the year
or (2) the second submission replaces the first submission". 

Because very few providers submit partial years, I will simplify the process, only keeping self-defined full year” cost reports. For our purposes, we define a “full year” as a report period having ten or more months. 

The other option would be to combine these reports with partial-year reporting periods, summing the appropriate fields, such as costs and revenue, to obtain a full year of data. We'd have to be cautious not to add certain numeric fields such as beds and FTEs since they are not meant to be accumulated.

Source for this information: Source: "HANDLING MULTIPLE REPORTS FOR A PROVIDER" here: https://www.lexjansen.com/sesug/2018/SESUG2018_Paper-287_Final_PDF.pdf

In [17]:
financials.shape

(12101, 65)

In [18]:
financials = financials[financials['Report Period (Months)'] >= 10]

In [19]:
financials.shape

(11702, 65)

In [20]:
(12116-11716)/12116

0.03301419610432486

We lose only about 3% of our dataset after filtering out short reporting periods.

Re-checking for duplicate hospital-years:

In [21]:
financials[financials[['Facility Name','Year']].duplicated(keep=False)][['Facility Name','Year','Fiscal Year Begin Date','Fiscal Year End Date','Report Period (Months)','Total Bad Debt expense','Total cost of charity care']].sort_values(by=['Facility Name','Year'])


,Facility Name,Year,Fiscal Year Begin Date,Fiscal Year End Date,Report Period (Months),Total Bad Debt expense,Total cost of charity care
1772,CHEROKEE MEDICAL CENTER,2020,2020-07-01,2021-06-30,11,6799176.0,383895.0
1778,CHEROKEE MEDICAL CENTER,2020,2019-10-01,2020-09-30,11,26836900.0,2368876.0
1774,CHEROKEE MEDICAL CENTER,2022,2022-01-01,2022-12-31,11,6108370.0,998609.0
1780,CHEROKEE MEDICAL CENTER,2022,2021-10-01,2022-09-30,11,16471735.0,2923591.0
1775,CHEROKEE MEDICAL CENTER,2023,2023-01-01,2023-12-31,11,NaN,NaN
...,...,...,...,...,...,...,...
11575,WAYNE MEMORIAL HOSPITAL,2022,2022-07-01,2023-06-30,11,9584116.0,553424.0
11561,WAYNE MEMORIAL HOSPITAL,2023,2023-07-01,2024-06-30,11,NaN,NaN
11576,WAYNE MEMORIAL HOSPITAL,2023,2023-07-01,2024-06-30,11,NaN,NaN
11562,WAYNE MEMORIAL HOSPITAL,2024,2024-07-01,2025-06-30,11,NaN,NaN


These duplicates likely result from a hospital updating or correcting the data initially reported. When this happens, it is common to use the most recently submitted (and likely most up-to-date) report and drop the old report to avoid double counting. That is what I will do here.

In [22]:
financials = (
    financials.sort_values(by=['Fiscal Year End Date'])
    .drop_duplicates(subset=['Facility Name', 'CCN','Year'], keep='last')
)

Re-checking for duplicate hospital-years:

In [23]:
financials[financials[['Facility Name','Year']].duplicated(keep=False)][['Facility Name','Year','Fiscal Year Begin Date','Fiscal Year End Date','Report Period (Months)','Total Bad Debt expense','Total cost of charity care']].sort_values(by=['Facility Name','Year'])


,Facility Name,Year,Fiscal Year Begin Date,Fiscal Year End Date,Report Period (Months),Total Bad Debt expense,Total cost of charity care
1778,CHEROKEE MEDICAL CENTER,2020,2019-10-01,2020-09-30,11,26836900.0,2368876.0
1772,CHEROKEE MEDICAL CENTER,2020,2020-07-01,2021-06-30,11,6799176.0,383895.0
1780,CHEROKEE MEDICAL CENTER,2022,2021-10-01,2022-09-30,11,16471735.0,2923591.0
1774,CHEROKEE MEDICAL CENTER,2022,2022-01-01,2022-12-31,11,6108370.0,998609.0
1781,CHEROKEE MEDICAL CENTER,2023,2022-10-01,2023-09-30,11,NaN,NaN
...,...,...,...,...,...,...,...
11560,WAYNE MEMORIAL HOSPITAL,2022,2022-07-01,2023-06-30,11,15706024.0,4265776.0
11561,WAYNE MEMORIAL HOSPITAL,2023,2023-07-01,2024-06-30,11,NaN,NaN
11576,WAYNE MEMORIAL HOSPITAL,2023,2023-07-01,2024-06-30,11,NaN,NaN
11562,WAYNE MEMORIAL HOSPITAL,2024,2024-07-01,2025-06-30,11,NaN,NaN


I've confirmed we have 1 row per hospital-year. Dropping the FY begin/end dates and the report period.

In [24]:
financials = financials.drop(columns=['Fiscal Year Begin Date','Fiscal Year End Date','Report Period (Months)'])

We want to make sure we still have rows for missing hospital-years, so that we can interpolate missing values later.

In [25]:
financials['Year'].min()

2010

In [26]:
financials['Year'].max()

2025

In [27]:
years = pd.Series(list(range(2010, 2026)), name='Year')
hospitals = hospitals_master[['Facility Name','CCN']]
hospital_years = pd.merge(years, hospitals, how="cross")
hospital_years.sort_values(by=['Facility Name','Year'])

,Year,Facility Name,CCN
0,2010,ACADIA GENERAL HOSPITAL,190044
846,2011,ACADIA GENERAL HOSPITAL,190044
1692,2012,ACADIA GENERAL HOSPITAL,190044
2538,2013,ACADIA GENERAL HOSPITAL,190044
3384,2014,ACADIA GENERAL HOSPITAL,190044
...,...,...,...
10151,2021,ZUNI COMPREHENSIVE COMMUNITY HEALTH CENTER,320060
10997,2022,ZUNI COMPREHENSIVE COMMUNITY HEALTH CENTER,320060
11843,2023,ZUNI COMPREHENSIVE COMMUNITY HEALTH CENTER,320060
12689,2024,ZUNI COMPREHENSIVE COMMUNITY HEALTH CENTER,320060


In [28]:
financials_full = hospital_years.merge(financials, how='left', on=['Facility Name','CCN','Year'])

Merging master hospital list with financial data:

In [29]:
hospitals_full = financials_full.drop(columns=['Facility Name']).merge(hospitals_master, how='right', on='CCN')

In [30]:
hospitals_full.sort_values(by=['Facility Name','Year'])[['Facility Name','CCN','Year','Financial Indicators: Net Profit Margin','Financial Indicators: SOLVENCY Debt Ratio']].head(10)


,Facility Name,CCN,Year,Financial Indicators: Net Profit Margin,Financial Indicators: SOLVENCY Debt Ratio
0,ACADIA GENERAL HOSPITAL,190044,2010,NaN,NaN
1,ACADIA GENERAL HOSPITAL,190044,2011,NaN,NaN
2,ACADIA GENERAL HOSPITAL,190044,2012,NaN,NaN
3,ACADIA GENERAL HOSPITAL,190044,2013,NaN,NaN
4,ACADIA GENERAL HOSPITAL,190044,2014,NaN,NaN
5,ACADIA GENERAL HOSPITAL,190044,2015,-0.113591,0.680679
6,ACADIA GENERAL HOSPITAL,190044,2016,-0.178962,0.455439
7,ACADIA GENERAL HOSPITAL,190044,2017,0.042598,0.462704
8,ACADIA GENERAL HOSPITAL,190044,2018,-0.043053,0.597040
9,ACADIA GENERAL HOSPITAL,190044,2019,0.013241,0.541594


We need to make sure and drop any financial data reported after a hospital scales down operations.

In [31]:
condition = (hospitals_full['Year'] < hospitals_full['Closure Date'].dt.year) | hospitals_full['Closure Date'].isna()
hospitals_full = hospitals_full[condition]

Creating a column that will be used as the indicator column for our predictive model. This will be a discrete multi-class integer column essentially answering the question "Did this facility close or scale down its operations within __ to __ years"? We will look at 3 different time windows: 0-1 year, 1-3 years, and 3-5 years. This will allow us to look at the strongest predictive factors for each class - something that will likely be helpful to hospital executives looking to identify longer-range risk factors for closure.

In [32]:
(hospitals_full['Closure Date'].dt.year - hospitals_full['Year']).value_counts()

1.0     31
2.0     30
4.0     29
3.0     29
5.0     28
6.0     27
10.0    26
9.0     26
8.0     26
7.0     26
11.0    22
12.0    20
13.0    16
14.0     9
15.0     5
16.0     1
Name: count, dtype: int64

In [36]:
diff = hospitals_full['Closure Date'].dt.year - hospitals_full['Year']

conditions = [
    (diff > 3) & (diff <= 5), # label:1
    (diff > 1) & (diff <= 3), # label:2
    (diff > 0) & (diff <= 1)  # label:3
]

choices = [1, 2, 3]

# Default is 0 (everything that falls through the cracks)
hospitals_full['Closure Proximity'] = np.select(conditions, choices, default=0)


In [37]:
hospitals_full[hospitals_full['Facility Name']=='ACOMA-CANONCITO-LAGUNA SERVICE UNIT'][['Facility Name','Year','Closed','Closure Date','Closure Proximity']]


,Facility Name,Year,Closed,Closure Date,Closure Proximity
32,ACOMA-CANONCITO-LAGUNA SERVICE UNIT,2010,1,2022-02-01,0
33,ACOMA-CANONCITO-LAGUNA SERVICE UNIT,2011,1,2022-02-01,0
34,ACOMA-CANONCITO-LAGUNA SERVICE UNIT,2012,1,2022-02-01,0
35,ACOMA-CANONCITO-LAGUNA SERVICE UNIT,2013,1,2022-02-01,0
36,ACOMA-CANONCITO-LAGUNA SERVICE UNIT,2014,1,2022-02-01,0
37,ACOMA-CANONCITO-LAGUNA SERVICE UNIT,2015,1,2022-02-01,0
38,ACOMA-CANONCITO-LAGUNA SERVICE UNIT,2016,1,2022-02-01,0
39,ACOMA-CANONCITO-LAGUNA SERVICE UNIT,2017,1,2022-02-01,1
40,ACOMA-CANONCITO-LAGUNA SERVICE UNIT,2018,1,2022-02-01,1
41,ACOMA-CANONCITO-LAGUNA SERVICE UNIT,2019,1,2022-02-01,2


In [37]:
hospitals_full.shape

(13391, 81)

In [38]:
len(hospitals_full['CCN'].unique())

846

In [39]:
hospitals_full.columns

Index(['Year', 'CCN', 'Hospital type', 'Control type', 'Medicaid charges',
       'HAC reduction adjustment amount',
       'STATEMENT OF REVENUES AND EXPENSES: Net Income (G3_C1_29)',
       'ADJUSTED SALARIES, Subtotal Salaries',
       'BALANCE SHEET: Prepaid expenses (G_C1THRU4_8)',
       'BED DAYS: Total Hospital', 'Number of Interns & Residents',
       'BALANCE SHEET: Cash on Hand and in Banks (G_C1THRU4_1)',
       'Cost To Charge Ratio', 'Total discharges',
       'REIMBURSEMENT SETTLEMENT: Payment to cost ratio',
       'Cost of Uncompensated Care',
       'BALANCE SHEET: Accounts Receivable (G_C1THRU4_4)', 'Total Salaries',
       'BALANCE SHEET: Total Assets (G_C1THRU4_36)',
       'BALANCE SHEET: Total Current Assets (G_C1THRU4_11)',
       'REIMBURSEMENT SETTLEMENT: Interim payments', 'Total Bad Debt expense',
       'TRIAL BALANCE OF EXPENSE ACCOUNTS: Interest Expense (A_C2_113)',
       'NUMBER OF BEDS: Adults & Pediatrics',
       'BALANCE SHEET: Temporary Investments

In [40]:
hospitals_full['Hospital Type'] = hospitals_full['Hospital Ownership'].combine_first(hospitals_full['Hospital type'])
hospitals_full = hospitals_full.drop(columns=['Hospital Ownership','Hospital type','Control type'])
                                              

### Gathering hospital quality data:

Hospital quality data includes the following:

Process of Care Scores

Shares of patients receiving evidence-based treatments for AMI, CHF, pneumonia, surgical care, and outpatient care. All-payer. 

HCAHPS

Average scores for the aggregated questions in HCAHPS patient satisfaction survey. All-payer. 

Mortality and Readmission 

Estimates of AMI, CHF, and pneumonia mortality and readmission rates. Medicare FFS patients only


Source: https://github.com/asacarny/hospital-compare/

In [41]:
# Exporting our ccns of interest in order to filter hospital quality data
pd.Series(hospitals_full['CCN'].unique()).to_csv('../data/ccns.txt',index=False)

POC data:

In [42]:
hosp_quality_poc = pd.read_csv('../hospital-compare-master/output/poc.csv').rename(columns={'pn':'CCN','year':'Year'})

In [43]:
hosp_quality_poc.columns

Index(['CCN', 'ami1_denom', 'ami2_denom', 'ami3_denom', 'ami4_denom',
       'ami5_denom', 'ami7_denom', 'ami8_denom', 'cac1_denom', 'cac2_denom',
       'cac3_denom', 'hf1_denom', 'hf2_denom', 'hf3_denom', 'hf4_denom',
       'op2_denom', 'op3_denom', 'op4_denom', 'op5_denom', 'op6_denom',
       'op7_denom', 'pn2_denom', 'pn3_denom', 'pn4_denom', 'pn5_denom',
       'pn6_denom', 'pn7_denom', 'scipcard2_denom', 'scipinf1_denom',
       'scipinf2_denom', 'scipinf3_denom', 'scipinf4_denom', 'scipinf6_denom',
       'scipinf9_denom', 'scipvte1_denom', 'scipvte2_denom', 'Year',
       'op2_share', 'op4_share', 'op6_share', 'ami10_denom', 'op1_denom',
       'scipinf10_denom', 'ami10_share', 'ami2_share', 'ami7_share',
       'ami8_share', 'cac1_share', 'cac2_share', 'cac3_share', 'hf1_share',
       'hf2_share', 'hf3_share', 'op7_share', 'pn3_share', 'pn6_share',
       'scipcard2_share', 'scipinf10_share', 'scipinf1_share',
       'scipinf2_share', 'scipinf3_share', 'scipinf4_share', 'sc

The meanings of these column names can be found in the crosswalk:

In [44]:
poc_xw = pd.read_csv('../hospital-compare-master/misc/poc_xwlk.csv')
poc_xw.head()

,measurecode,condition,measurename
0,ami1,Heart Attack,Patients Given Aspirin at Arrival
1,ami2,Heart Attack,Patients Given Aspirin at Discharge
2,ami3,Heart Attack,Patients Given ACE Inhibitor for Left Ventricu...
3,ami3,Heart Attack,Patients Given ACE Inhibitor or ARB for Left V...
4,ami4,Heart Attack,Patients Given Adult Smoking Cessation Advice/...


HCAHPS data:

In [45]:
hosp_quality_hcahps = pd.read_csv('../hospital-compare-master/output/hcahps.csv').rename(columns={'pn':'CCN','year':'Year'})

In [46]:
hosp_quality_hcahps.columns

Index(['CCN', 'nsurveys', 'rrate', 'clean_score', 'commdoc_score',
       'commnurse_score', 'explain_score', 'help_score', 'info_score',
       'overall_score', 'pain_score', 'quiet_score', 'recommend_score', 'Year',
       'understood_score'],
      dtype='object')

The meanings of these column names can be found in the crosswalk:

In [47]:
hcahps_xw = pd.read_csv('../hospital-compare-master/misc/hcahps_xwlk.csv')
hcahps_xw.head()

,questionno,question,answerdescription,measurecode,meas,value,note
0,1,How do patients rate the hospital overall?,Patients who gave a rating of 6 or lower (low),H_HSP_RATING_0_6,overall,1,NaN
1,1,How do patients rate the hospital overall?,Patients who gave a rating of 7 or 8 (medium),H_HSP_RATING_7_8,overall,2,NaN
2,1,How do patients rate the hospital overall?,Patients who gave a rating of 9 or 10 (high),H_HSP_RATING_9_10,overall,3,NaN
3,2,How often did doctors communicate well with pa...,Doctors always communicated well,H_COMP_2_A_P,commdoc,3,NaN
4,2,How often did doctors communicate well with pa...,Doctors sometimes or never communicated well,H_COMP_2_SN_P,commdoc,1,NaN


In [48]:
hcahps_xw_cb = pd.read_csv('../hospital-compare-master/misc/hcahps_xwlk_codebook.csv')
hcahps_xw_cb.head()

,meas,question,answerdescription,value,note
0,overall,How do patients rate the hospital overall?,Patients who gave a rating of 9 or 10 (high),3,NaN
1,overall,How do patients rate the hospital overall?,Patients who gave a rating of 7 or 8 (medium),2,NaN
2,overall,How do patients rate the hospital overall?,Patients who gave a rating of 6 or lower (low),1,NaN
3,commdoc,How often did doctors communicate well with pa...,Doctors always communicated well,3,NaN
4,commdoc,How often did doctors communicate well with pa...,Doctors usually communicated well,2,NaN


Mortality and Readmission data:

In [49]:
hosp_quality_mort = pd.read_csv('../hospital-compare-master/output/mortreadm.csv').rename(columns={'pn':'CCN','year':'Year'})

In [50]:
hosp_quality_mort.columns

Index(['CCN', 'ami_mort_rate', 'ami_readm_rate', 'hf_mort_rate',
       'hf_readm_rate', 'pn_mort_rate', 'pn_readm_rate', 'ami_mort_npatients',
       'ami_readm_npatients', 'hf_mort_npatients', 'hf_readm_npatients',
       'pn_mort_npatients', 'pn_readm_npatients', 'Year', 'all_readm_rate',
       'hk_readm_rate', 'all_readm_npatients', 'hk_readm_npatients',
       'copd_mort_rate', 'copd_readm_rate', 'stk_mort_rate', 'stk_readm_rate',
       'copd_mort_npatients', 'copd_readm_npatients', 'stk_mort_npatients',
       'stk_readm_npatients', 'cabg_mort_rate', 'cabg_readm_rate',
       'cabg_mort_npatients', 'cabg_readm_npatients'],
      dtype='object')

Merging all hospital quality data together, then merging into the main dataset:

In [51]:
hosp_quality_poc['CCN'] = hosp_quality_poc['CCN'].astype(object)
hosp_quality_hcahps['CCN'] = hosp_quality_hcahps['CCN'].astype(object)
hosp_quality_mort['CCN'] = hosp_quality_mort['CCN'].astype(object)

In [52]:
intermediate = hosp_quality_poc.merge(hosp_quality_hcahps, how='outer', on=['CCN','Year'])

In [53]:
hosp_quality_full = intermediate.merge(hosp_quality_mort, how='outer', on=['CCN','Year'])

In [54]:
hospitals_full = hospitals_full.merge(hosp_quality_full, how='left', on=['CCN','Year'])

In [55]:
hospitals_full.shape

(13391, 183)

In [56]:
len(hospitals_full['CCN'].unique())

846

### Gathering health resource/demographic data:

I'll get health resource and demographic data from the Area Health Resources Files (publicly available in the HRSA/Health Resources and Services Administration data warehouse: https://data.hrsa.gov/data/download). 

This dataset provides current as well as historic data for more than 6,000 variables for each of the nation’s counties, as well as state and national data. It contains information on health facilities, health professions, measures of resource scarcity, health status, economic activity, health training programs, and socioeconomic and environmental characteristics. 

In [57]:
ARHF_Full = pd.read_csv('../data/ARHF_Full.csv')

In [58]:
ARHF_Full['County_FIPS'] = ARHF_Full['County_FIPS'].astype(int).astype(str).str.zfill(3)
ARHF_Full['State_FIPS'] = ARHF_Full['State_FIPS'].astype(int).astype(str).str.zfill(2)


In [59]:
ARHF_Full[['State_FIPS','County_FIPS','State','County']]

,State_FIPS,County_FIPS,State,County
0,10,005,DE,SUSSEX
1,10,005,DE,SUSSEX
2,10,005,DE,SUSSEX
3,10,005,DE,SUSSEX
4,10,005,DE,SUSSEX
...,...,...,...,...
8905,56,039,WY,TETON
8906,56,039,WY,TETON
8907,56,039,WY,TETON
8908,56,039,WY,TETON


In [60]:
ARHF_Full[['State_FIPS','County_FIPS','State','County']].dtypes

State_FIPS     object
County_FIPS    object
State          object
County         object
dtype: object

In [61]:
# Standardizing merge columns
hospitals_full['State_FIPS'] = hospitals_full['State_FIPS'].astype(int).astype(str).str.zfill(2)
hospitals_full['County_FIPS'] = hospitals_full['County_FIPS'].astype(str)
hospitals_full['County_FIPS'] = hospitals_full['County_FIPS'].str[2:]

In [62]:
hospitals_full[['State_FIPS','County_FIPS','State','County']]

,State_FIPS,County_FIPS,State,County
0,22,001,LA,ACADIA
1,22,001,LA,ACADIA
2,22,001,LA,ACADIA
3,22,001,LA,ACADIA
4,22,001,LA,ACADIA
...,...,...,...,...
13386,35,031,NM,MCKINLEY
13387,35,031,NM,MCKINLEY
13388,35,031,NM,MCKINLEY
13389,35,031,NM,MCKINLEY


In [63]:
hospitals_full[['State_FIPS','County_FIPS','State','County']].dtypes

State_FIPS     object
County_FIPS    object
State          object
County         object
dtype: object

In [64]:
hospitals_full = hospitals_full.merge(ARHF_Full, how='left', on=['State_FIPS','County_FIPS','State','County','Year'])

In [65]:
hospitals_full.head()

,Year,CCN,Medicaid charges,HAC reduction adjustment amount,STATEMENT OF REVENUES AND EXPENSES: Net Income (G3_C1_29),"ADJUSTED SALARIES, Subtotal Salaries",BALANCE SHEET: Prepaid expenses (G_C1THRU4_8),BED DAYS: Total Hospital,Number of Interns & Residents,BALANCE SHEET: Cash on Hand and in Banks (G_C1THRU4_1),...,"Per Capita Phys,Primary Care, Patient Care Non-Fed",Per Capita Short Term Gen Hosp Admissions,Per Capita Short Term General Hosp Beds,Per Capita Total Active D.O.s Non-Federal,Per Capita Total Active M.D.s Non-Federal,Per Capita Total Medicare Inpatient Days Short Term General Hospitals,Per Capita Total Number Hospitals,Percent Persons in Poverty,Population Estimate,"Unemployment Rate, 16+"
0,2010,190044,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.000340,0.092921,0.003529,0.0,0.000664,0.241335,0.000065,21.0,61773.0,6.8
1,2011,190044,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.000371,0.085783,0.003275,0.0,0.000710,0.230389,0.000065,22.4,61982.0,6.3
2,2012,190044,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.000355,0.083247,0.003279,0.0,0.000695,0.244573,0.000065,20.7,61912.0,5.6
3,2013,190044,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.000354,0.082776,0.003263,0.0,0.000675,0.262877,0.000064,19.0,62204.0,NaN
4,2014,190044,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.000368,0.082386,0.003249,0.0,0.000736,0.269148,0.000048,22.0,62486.0,5.7


In [66]:
hospitals_full.shape

(13391, 204)

In [67]:
print(hospitals_full.columns.tolist())

['Year', 'CCN', 'Medicaid charges', 'HAC reduction adjustment amount', 'STATEMENT OF REVENUES AND EXPENSES: Net Income (G3_C1_29)', 'ADJUSTED SALARIES, Subtotal Salaries', 'BALANCE SHEET: Prepaid expenses (G_C1THRU4_8)', 'BED DAYS: Total Hospital', 'Number of Interns & Residents', 'BALANCE SHEET: Cash on Hand and in Banks (G_C1THRU4_1)', 'Cost To Charge Ratio', 'Total discharges', 'REIMBURSEMENT SETTLEMENT: Payment to cost ratio', 'Cost of Uncompensated Care', 'BALANCE SHEET: Accounts Receivable (G_C1THRU4_4)', 'Total Salaries', 'BALANCE SHEET: Total Assets (G_C1THRU4_36)', 'BALANCE SHEET: Total Current Assets (G_C1THRU4_11)', 'REIMBURSEMENT SETTLEMENT: Interim payments', 'Total Bad Debt expense', 'TRIAL BALANCE OF EXPENSE ACCOUNTS: Interest Expense (A_C2_113)', 'NUMBER OF BEDS: Adults & Pediatrics', 'BALANCE SHEET: Temporary Investments (G_C1THRU4_2)', 'NUMBER OF BEDS: Total Hospital', 'Total Inpatient Days', 'Total Days Title XVIII', 'Total cost of charity care', 'Total Costs', 'NUMB

In [68]:
hospitals_full.isna().sum()

Year                                                                         0
CCN                                                                          0
Medicaid charges                                                          8964
HAC reduction adjustment amount                                          12355
STATEMENT OF REVENUES AND EXPENSES: Net Income (G3_C1_29)                 7817
                                                                         ...  
Per Capita Total Medicare Inpatient Days Short Term General Hospitals     5103
Per Capita Total Number Hospitals                                         5103
Percent Persons in Poverty                                                4419
Population Estimate                                                       3629
Unemployment Rate, 16+                                                    4398
Length: 204, dtype: int64

In [69]:
hospitals_full['County_FIPS'].isna().sum()

np.int64(0)

In [70]:
# Confirming one hospital per year
hospitals_full[hospitals_full[['CCN','Year']].duplicated(keep=False)]

,Year,CCN,Medicaid charges,HAC reduction adjustment amount,STATEMENT OF REVENUES AND EXPENSES: Net Income (G3_C1_29),"ADJUSTED SALARIES, Subtotal Salaries",BALANCE SHEET: Prepaid expenses (G_C1THRU4_8),BED DAYS: Total Hospital,Number of Interns & Residents,BALANCE SHEET: Cash on Hand and in Banks (G_C1THRU4_1),...,"Per Capita Phys,Primary Care, Patient Care Non-Fed",Per Capita Short Term Gen Hosp Admissions,Per Capita Short Term General Hosp Beds,Per Capita Total Active D.O.s Non-Federal,Per Capita Total Active M.D.s Non-Federal,Per Capita Total Medicare Inpatient Days Short Term General Hospitals,Per Capita Total Number Hospitals,Percent Persons in Poverty,Population Estimate,"Unemployment Rate, 16+"


In [71]:
len(hospitals_full['CCN'].unique())

846

Adjusting financial data for inflation (converting financial values into 2025 purchasing power using CPI index):

In [72]:
cols_to_update = ['BALANCE SHEET: Total Assets (G_C1THRU4_36)',
                  'REIMBURSEMENT SETTLEMENT: Interim payments', 
                  'HAC reduction adjustment amount', 
                  'IPPS Interim payment', 
                  'BALANCE SHEET: Cash on Hand and in Banks (G_C1THRU4_1)', 
                  'BALANCE SHEET: Total Current Liabilities (G_C1THRU4_45)', 
                  'Total cost of charity care', 
                  'HVBP payment adjustment amount', 
                  'Total Salaries', 
                  'BALANCE SHEET: Accounts Receivable (G_C1THRU4_4)', 
                  'Net Revenue from Medicaid', 
                  'Total Bad Debt expense', 
                  'Total Charges', 
                  'ADJUSTED SALARIES, Subtotal Salaries', 
                  'Donations, Land Improvements', 
                  'Total Costs', 
                  'BALANCE SHEET: Prepaid expenses (G_C1THRU4_8)', 
                  'BALANCE SHEET: Total Current Assets (G_C1THRU4_11)', 
                  'REIMBURSEMENT SETTLEMENT: Subtotal', 
                   'RECONCILIATION OF CAPITAL COST CENTERS: Depreciation, Total (A7_3_C9_3)', 
                    'STATEMENT OF REVENUES AND EXPENSES: Net Patient Revenue (G3_C1_3)', 
                  'STATEMENT OF REVENUES AND EXPENSES: Net Income (G3_C1_29)', 
                  'BALANCE SHEET: Inventory (G_C1THRU4_7)', 
                  'IPPS Payment amount (unadjusted)', 
                  'BALANCE SHEET: Temporary Investments (G_C1THRU4_2)', 
                  'TRIAL BALANCE OF EXPENSE ACCOUNTS: Interest Expense (A_C2_113)', 
                  'Cost of Uncompensated Care', 
                  'Medicaid charges', 
                  'Total Liabilities',
                  'Financial Indicators: Total Net Assets', 
                  'S-10 DATA: Medicaid Costs', 
                  'Financial Indicators: Cash Reserves', 
                  'Financial Indicators: Net Profit Margin', 
                  'Financial Indicators: Operating Profit', 
                  'Financial Indicators: Operating Profit Margin', 
                  'Financial Indicators: SOLVENCY total assets less total liabilities', 
                  'Median Household Income',
                  'Per Capita Personal Income'
                 ]

In [73]:
cpi = pd.read_excel('../data/CPI_Data.xlsx',skiprows=11)[['Year','Annual']]

/opt/anaconda3/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [74]:
cpi

,Year,Annual
0,2010,218.056
1,2011,224.939
2,2012,229.594
3,2013,232.957
4,2014,236.736
5,2015,237.017
6,2016,240.007
7,2017,245.120
8,2018,251.107
9,2019,255.657


In [75]:
cpi.dtypes

Year        int64
Annual    float64
dtype: object

In [76]:
cpi[cpi['Year']==2025]['Annual']

15    321.943
Name: Annual, dtype: float64

In [77]:
cpi['Scale Factor'] = cpi[cpi['Year']==2025]['Annual'].item() / cpi['Annual']

In [78]:
cpi

,Year,Annual,Scale Factor
0,2010,218.056,1.476423
1,2011,224.939,1.431246
2,2012,229.594,1.402227
3,2013,232.957,1.381985
4,2014,236.736,1.359924
5,2015,237.017,1.358312
6,2016,240.007,1.341390
7,2017,245.120,1.313410
8,2018,251.107,1.282095
9,2019,255.657,1.259277


In [79]:
cpi_dict = dict(zip(cpi["Year"], cpi["Scale Factor"]))

In [80]:
# Map the year column to the CPI scale factor using the dictionary
cpi_series = hospitals_full['Year'].map(cpi_dict)

# Multiply the target columns by the CPI series
for col in cols_to_update:
    hospitals_full[col] = hospitals_full[col] * cpi_series

In [81]:
# Save full dataset
hospitals_full.to_csv('../data/hospitals_full.csv', index=False)